## Étape 1 — Littéraux $L$

$$L = \{a,b,c,q,p,r,s,o\}$$


In [8]:
L = {"a", "b", "c", "q", "p", "r", "s", "o"}

print("L =", L)
print("b in L ?", "b" in L)
print("w in L ?", "w" in L)
assert len(L) == 8

L = {'c', 'p', 's', 'r', 'o', 'a', 'q', 'b'}
b in L ? True
w in L ? False


## Étape 2 — Hypothèses $A$

$$A = \{a,b,c\}$$

In [9]:
A = {"a", "b", "c"}

print("A =", A)
print("A ⊆ L ?", A.issubset(L))
assert A <= L

A = {'c', 'b', 'a'}
A ⊆ L ? True


## Étape 3 — Contraires

$$\bar a = r,\quad \bar b = s,\quad \bar c = o$$

In [11]:
contraries = {"a": "r", "b": "s", "c": "o"}

print("contrary(a) =", contraries["a"])
assert contraries["a"] == "r"
assert set(contraries) == A
assert set(contraries.values()) <= L

contrary(a) = r


## Étape 4 — Règles $R$

| Rule | Conclusion | Premises |
|------|------------|----------|
| $r_1$ | $p$ | $q, a$ |
| $r_2$ | $q$ | *(none — fact)* |
| $r_3$ | $r$ | $b, c$ |
| $r_4$ | $o$ | $p, c$ |
| $r_5$ | $s$ | $o$ |


In [13]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(r"C:\work univ\projet DYNC\aba-part-dync-project")
sys.path.insert(0, str(PROJECT_ROOT))

from backend.aba_framework import Rule
R = [
    Rule(conclusion="p", premises=("q", "a"), name="r1"),
    Rule(conclusion="q", premises=(), name="r2"),  # fact
    Rule(conclusion="r", premises=("b", "c"), name="r3"),
    Rule(conclusion="o", premises=("p", "c"), name="r4"),
    Rule(conclusion="s", premises=("o",), name="r5"),
]

for rule in R:
    print(rule)

assert R[1].conclusion == "q" and R[1].premises == ()

r1: p <- q, a
r2: q <-
r3: r <- b, c
r4: o <- p, c
r5: s <- o


## Étape 5 — Assembler le framework

In [14]:
import importlib
import backend.aba_framework as aba_framework

aba_framework = importlib.reload(aba_framework)
ABAFramework = aba_framework.ABAFramework

fw = ABAFramework.from_components(L, A, contraries, R)
print(fw.summary())
print("\nFlat?", fw.isFlat())

ABA Framework
  L = {a, b, c, p, q, r, s, o}
  A = {a, b, c}
  Contraries:
    contrary(a) = r
    contrary(b) = s
    contrary(c) = o
  Rules:
    r1: p <- q, a
    r2: q <-
    r3: r <- b, c
    r4: o <- p, c
    r5: s <- o

Flat? True


## Étape 6 — Checklist de validation

- [x] 8 literals
- [x] 3 assumptions ⊆ L
- [x] 3 contraries
- [x] 5 rules
- [x] fact $q \leftarrow$
- [x] all rule literals ∈ L

In [15]:
assert len(fw.literals) == 8
assert fw.assumptions <= fw.literals and len(fw.assumptions) == 3
assert len(fw.contraries) == 3
assert len(fw.rules) == 5
assert any(r.conclusion == "q" and r.premises == () for r in fw.rules)
for r in fw.rules:
    assert r.conclusion in fw.literals
    assert all(p in fw.literals for p in r.premises)
print("All validation checks passed.")

All validation checks passed.


## Conversion: non-circular & atomic

In [16]:
print("Circular dependencies?", fw.contains_circular_dependenncies())
fw_nc = fw.to_non_circular()
print("After to_non_circular, circular?", fw_nc.contains_circular_dependenncies())

print("\nOriginally atomic?", fw.isAtomic())
fw_at = fw.make_atomic()
print("After to_atomic, atomic?", fw_at.isAtomic())
print(fw_at.summary())

Circular dependencies? False
After to_non_circular, circular? False

Originally atomic? False
After to_atomic, atomic? True
ABA Framework
  L = {a, a_r1, a_r3, a_r4, a_r5, b, c, fail_r1, fail_r3, fail_r4, fail_r5, p, q, r, s, o}
  A = {a, a_r1, a_r3, a_r4, a_r5, b, c}
  Contraries:
    contrary(a) = r
    contrary(a_r1) = fail_r1
    contrary(a_r3) = fail_r3
    contrary(a_r4) = fail_r4
    contrary(a_r5) = fail_r5
    contrary(b) = s
    contrary(c) = o
  Rules:
    r1: p <- a_r1
    r2: q <-
    r3: r <- a_r3
    r4: o <- a_r4
    r5: s <- a_r5


## Arguments

An argument is a claim deduced from a set of assumptions using $R$.
Assumptions themselves are trivial arguments.

In [17]:
args = fw.create_arguments()
print(f"{len(args)} arguments:\n")
for arg in args:
    print(arg)

8 arguments:

A1: a <- {a}
A2: b <- {b}
A3: c <- {c}
A4: p <- {a}
A5: q <- {}
A6: r <- {b, c}
A7: s <- {a, c}
A8: o <- {a, c}


## Classical attacks

$X$ attacks $Y$ on assumption $\alpha$ iff $\mathrm{claim}(X) = \bar\alpha$ and $\alpha \in \mathrm{assumptions}(Y)$.

In [18]:
attacks = fw.attacks_classic(args)
print(f"{len(attacks)} classical attacks:\n")
for x, y, alpha in attacks:
    print(f"{x.name} -> {y.name}  (on {alpha})   [{x.claim} <- {set(x.assumptions)}]")

10 classical attacks:

A6 -> A1  (on a)   [r <- {'b', 'c'}]
A6 -> A4  (on a)   [r <- {'b', 'c'}]
A6 -> A7  (on a)   [r <- {'b', 'c'}]
A6 -> A8  (on a)   [r <- {'b', 'c'}]
A7 -> A2  (on b)   [s <- {'c', 'a'}]
A7 -> A6  (on b)   [s <- {'c', 'a'}]
A8 -> A3  (on c)   [o <- {'c', 'a'}]
A8 -> A6  (on c)   [o <- {'c', 'a'}]
A8 -> A7  (on c)   [o <- {'c', 'a'}]
A8 -> A8  (on c)   [o <- {'c', 'a'}]


## Preferences — normal / reverse attacks (ABA+)

In [20]:

from backend.aba_framework import create_example


fw_pref = create_example(preferences={"b": {"a"}, "c": {"a"}})
print(fw_pref.summary())

res = fw_pref.normal_reverse_attacks()
print(f"\nNormal attacks ({len(res['normal'])}):")
for x, y, alpha in res["normal"]:
    print(f"  {x.name} -> {y.name} (on {alpha})")

print(f"\nReverse attacks ({len(res['reverse'])}):")
for x, y, alpha in res["reverse"]:
    print(f"  {x.name} -> {y.name} (on {alpha})")

ABA Framework
  L = {a, b, c, p, q, r, s, o}
  A = {a, b, c}
  Contraries:
    contrary(a) = r
    contrary(b) = s
    contrary(c) = o
  Rules:
    r1: p <- q, a
    r2: q <-
    r3: r <- b, c
    r4: o <- p, c
    r5: s <- o

Normal attacks (10):
  A6 -> A1 (on a)
  A6 -> A4 (on a)
  A6 -> A7 (on a)
  A6 -> A8 (on a)
  A7 -> A2 (on b)
  A7 -> A6 (on b)
  A8 -> A3 (on c)
  A8 -> A6 (on c)
  A8 -> A7 (on c)
  A8 -> A8 (on c)

Reverse attacks (0):


## Attack graph

In [22]:
import matplotlib.pyplot as plt
import networkx as nx

G = nx.DiGraph()
for arg in args:
    G.add_node(arg.name, label=f"{arg.name}\n{arg.claim}")
for x, y, alpha in attacks:
    G.add_edge(x.name, y.name, label=alpha)

plt.figure(figsize=(10, 7))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_size=1400, node_color="#dceaf7", arrows=True)
edge_labels = {(u, v): d["label"] for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)
plt.title("Classical ABA attack graph (example)")
plt.axis("off")
plt.tight_layout()
plt.show()

C:\Users\Nour\AppData\Local\Temp\ipykernel_20948\649350684.py:17: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
